# Game Intelligence AI — 01: Environment Setup

**الهدف:** تجهيز بيئة Google Colab، فحص GPU والذاكرة والتخزين، ربط Google Drive، تثبيت مكتبات المشروع بإصدارات محددة، فحص Hugging Face، وإنشاء هيكل المجلدات وحفظ تقرير البيئة.

> **مهم قبل البدء**
> 1. افتح هذه النسخة في Google Colab.
> 2. من القائمة اختر **Runtime → Change runtime type → T4 GPU** إذا كان متاحًا. لا توجد ضمانة أن Colab المجاني سيوفر GPU معينًا.
> 3. شغّل الخلايا بالترتيب من الأعلى إلى الأسفل باستخدام زر التشغيل بجانب كل خلية أو **Runtime → Run all**.
> 4. عند طلب الإذن، وافق على ربط Google Drive.
> 5. لا تضع كلمة مرور أو Hugging Face Token داخل الكود، ولا تشارك نسخة تحتوي على أسرار.

**هذه المرحلة لا تبدأ تدريب النموذج ولا تنزّل أوزانه الكبيرة.** هي فقط تتحقق من أن البيئة جاهزة وتُنشئ الملفات الأساسية.

## الخلية 1 — معلومات الجلسة

تعرض إصدار Python ومعلومات نظام التشغيل. لا تغيّر أي إعدادات.

In [ ]:
import sys, os, platform, time
from datetime import datetime, timezone

print("Game Intelligence AI — Environment Setup")
print("Local runtime time (UTC):", datetime.now(timezone.utc).isoformat())
print("Python:", sys.version)
print("Platform:", platform.platform())
print("Working directory:", os.getcwd())

## الخلية 2 — فحص GPU وذاكرة CUDA

تأكد أن نوع Runtime مضبوط على GPU. إذا ظهر `CUDA available: False` فهذا يعني أن الجلسة تعمل على CPU؛ يمكنك متابعة إعداد الملفات، لكن لا تبدأ تنزيل/تدريب نموذج 7B قبل توفير GPU مناسب.

الخلية لا تحجز ذاكرة كبيرة؛ تنفذ اختبارًا صغيرًا ثم تحرر الذاكرة.

In [ ]:
import subprocess, shutil

try:
    import torch
    print("PyTorch version:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    print("PyTorch CUDA build:", torch.version.cuda)
    if torch.cuda.is_available():
        print("GPU count:", torch.cuda.device_count())
        for i in range(torch.cuda.device_count()):
            props = torch.cuda.get_device_properties(i)
            total_gib = props.total_memory / (1024**3)
            print(f"GPU {i}: {props.name}")
            print(f"  Total VRAM: {total_gib:.2f} GiB")
            print(f"  Compute capability: {props.major}.{props.minor}")
        free_b, total_b = torch.cuda.mem_get_info()
        print(f"Current free VRAM: {free_b/(1024**3):.2f} GiB")
        print(f"Current total VRAM: {total_b/(1024**3):.2f} GiB")
        # Small smoke test (about 16 MiB for float32)
        test_tensor = torch.zeros((1024, 1024), device="cuda")
        print("Small CUDA allocation test: PASSED")
        del test_tensor
        torch.cuda.empty_cache()
    else:
        print("ACTION: Runtime > Change runtime type > GPU, then reconnect.")
except Exception as e:
    print("GPU check failed:", repr(e))

print("\nNVIDIA-SMI:")
if shutil.which("nvidia-smi"):
    result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
    print(result.stdout[:6000] if result.stdout else result.stderr)
else:
    print("nvidia-smi not found (normal when no NVIDIA GPU is attached).")

## الخلية 3 — ربط Google Drive

سيُحفظ المشروع والتقارير والنسخ الاحتياطية في Drive حتى لا تضيع عند انتهاء جلسة Colab.

عند ظهور نافذة الصلاحيات، اختر حساب Google الصحيح ووافق. إذا فشل الربط، أعد تشغيل الخلية مرة واحدة بعد إعادة الاتصال.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT = "/content/drive/MyDrive"
PROJECT_NAME = "GameIntelligenceAI"
PROJECT_ROOT = os.path.join(DRIVE_ROOT, PROJECT_NAME)

print("Drive mounted:", os.path.isdir(DRIVE_ROOT))
print("Project root:", PROJECT_ROOT)

## الخلية 4 — فحص مساحة التخزين

تعرض المساحة المتاحة في مساحة Colab المؤقتة وفي Google Drive (إذا كان النظام يسمح بإظهارها عبر `disk_usage`). لا تعتمد على مساحة Colab المؤقتة لحفظ ملفات مهمة؛ استخدم Drive.

In [ ]:
import shutil

def show_disk(label, path):
    try:
        usage = shutil.disk_usage(path)
        print(f"{label}: total={usage.total/(1024**3):.2f} GiB | "
              f"used={usage.used/(1024**3):.2f} GiB | "
              f"free={usage.free/(1024**3):.2f} GiB")
    except Exception as exc:
        print(f"{label}: unable to inspect ({exc})")

show_disk("Colab local storage", "/content")
show_disk("Google Drive", DRIVE_ROOT)

## الخلية 5 — تثبيت مكتبات المشروع بإصدارات محددة

**ملاحظة توافق مهمة:** لا نحدّث PyTorch أو CUDA تلقائيًا؛ Colab يجهز نسخة متوافقة مع GPU/runtime الخاص به. نثبت طبقة أدوات Hugging Face وQLoRA وواجهة العرض بإصدارات محددة لتقليل تغيّر البيئة بين الجلسات.

إذا ظهر تنبيه يطلب إعادة تشغيل Runtime بعد التثبيت:
1. اختر **Runtime → Restart session**.
2. بعد إعادة التشغيل، أعد تشغيل الخلايا 1 إلى 4 لإعادة تعريف متغيرات الجلسة وربط Drive، ثم تخطَّ الخلية 5 وانتقل إلى الخلية 6. لا حاجة لإعادة تشغيل خلية التثبيت إلا إذا فشل التثبيت.

قد تتغير توافقات الحزم مستقبلًا؛ إذا أظهر pip تعارضًا، احتفظ بنص الخطأ ولا تثبّت حزمًا عشوائية فوقها.

In [ ]:
%pip install -q \\
  "transformers==4.57.1" \\
  "datasets==4.4.1" \\
  "accelerate==1.11.0" \\
  "peft==0.18.0" \\
  "trl==0.24.0" \\
  "bitsandbytes==0.48.1" \\
  "huggingface_hub==0.36.0" \\
  "safetensors==0.6.2" \\
  "sentence-transformers==5.1.2" \\
  "faiss-cpu==1.12.0" \\
  "gradio==5.49.1" \\
  "pytest==8.4.2" \\
  "jsonlines==4.0.0"

## الخلية 6 — التحقق من المكتبات بعد التثبيت

إذا أعدت تشغيل Runtime، شغّل هذه الخلية بعد إعادة الاتصال. ستعرض الإصدارات الفعلية وتلتقط أي مكتبة لم تُحمّل.

In [ ]:
import importlib
from importlib.metadata import version, PackageNotFoundError

packages = [
    "torch", "transformers", "datasets", "accelerate", "peft", "trl",
    "bitsandbytes", "huggingface_hub", "safetensors",
    "sentence-transformers", "faiss-cpu", "gradio", "pytest", "jsonlines"
]

package_status = {}
for package in packages:
    try:
        package_status[package] = version(package)
        print(f"[OK] {package}: {package_status[package]}")
    except PackageNotFoundError:
        package_status[package] = None
        print(f"[MISSING] {package}")

# Import smoke tests for important libraries
imports = [
    ("torch", "torch"),
    ("transformers", "transformers"),
    ("datasets", "datasets"),
    ("accelerate", "accelerate"),
    ("peft", "peft"),
    ("trl", "trl"),
    ("bitsandbytes", "bitsandbytes"),
    ("huggingface_hub", "huggingface_hub"),
    ("sentence_transformers", "sentence_transformers"),
    ("faiss", "faiss"),
    ("gradio", "gradio"),
]
import_status = {}
for module_name, label in imports:
    try:
        importlib.import_module(module_name)
        import_status[label] = "OK"
        print(f"[IMPORT OK] {label}")
    except Exception as exc:
        import_status[label] = f"ERROR: {type(exc).__name__}: {exc}"
        print(f"[IMPORT ERROR] {label}: {exc}")

## الخلية 7 — إعداد Hugging Face وفحص الوصول

سنفحص صفحة النموذج دون تنزيل الأوزان. النموذج التجريبي الافتراضي هو `Qwen/Qwen2.5-Coder-7B-Instruct`، ويمكن تغيير اسمه لاحقًا بعد مرحلة المقارنة.

**الأمان:**
- لا تكتب Token داخل الخلية.
- إذا احتجت Token، أضفه في Colab من رمز المفتاح **Secrets** باسم `HF_TOKEN`، وفعّل وصول Notebook إليه.
- لا تطبع قيمة Token ولا تحفظها في GitHub أو ملفات المشروع.
- فحص النموذج هنا لا يعني أننا قررنا نهائيًا أنه أفضل نموذج عالميًا؛ سنقارن المرشحين في Notebook مستقل.

In [ ]:
MODEL_ID = "Qwen/Qwen2.5-Coder-7B-Instruct"

HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    pass

try:
    from huggingface_hub import HfApi, login
    if HF_TOKEN:
        login(token=HF_TOKEN, add_to_git_credential=False)
        print("Hugging Face authentication: token found in Colab Secrets.")
    else:
        print("Hugging Face authentication: no token found; testing public access anonymously.")

    api = HfApi(token=HF_TOKEN)
    info = api.model_info(MODEL_ID)
    print("Model repository: accessible")
    print("Model ID:", info.id)
    print("Pipeline tag:", getattr(info, "pipeline_tag", None))
    print("Last modified:", getattr(info, "last_modified", None))
    print("Gated:", getattr(info, "gated", None))
except Exception as exc:
    print("Hugging Face check failed:", repr(exc))
    print("Check internet connection, model ID, and whether the repository requires approval.")

## الخلية 8 — إنشاء هيكل مجلدات المشروع

تنشئ هذه الخلية المجلدات الأساسية داخل Google Drive. لا تحذف أي ملفات موجودة ولا تستبدلها.

In [ ]:
from pathlib import Path

DIRECTORIES = [
    "notebooks",
    "datasets/raw",
    "datasets/processed",
    "datasets/train",
    "datasets/validation",
    "datasets/test",
    "models/base",
    "models/adapters",
    "models/releases",
    "knowledge_base/godot",
    "knowledge_base/unity",
    "knowledge_base/unreal",
    "knowledge_base/game_programming",
    "agents/architect",
    "agents/programmer",
    "agents/designer",
    "agents/tester",
    "agents/orchestrator",
    "benchmarks",
    "evaluation",
    "app",
    "configs",
    "documentation",
    "logs",
    "backups"
]

for relative_path in DIRECTORIES:
    (Path(PROJECT_ROOT) / relative_path).mkdir(parents=True, exist_ok=True)

print(f"Created/verified {len(DIRECTORIES)} directories.")
print("Project root:", PROJECT_ROOT)

## الخلية 9 — فحص صلاحية الكتابة إلى Drive

تنشئ ملف اختبار صغيرًا ثم تقرأه وتحذفه. إذا نجح الاختبار، فهذا يؤكد أن Notebook يستطيع الكتابة إلى مجلد المشروع.

In [ ]:
from pathlib import Path

test_file = Path(PROJECT_ROOT) / "logs" / "_drive_write_test.txt"
test_file.write_text("Game Intelligence AI Drive write test: OK\n", encoding="utf-8")
read_back = test_file.read_text(encoding="utf-8")
assert "OK" in read_back
test_file.unlink()
print("Google Drive write/read/delete test: PASSED")

## الخلية 10 — إنشاء تقرير البيئة وحفظه

تحفظ تقرير JSON وTXT في `GameIntelligenceAI/documentation/`. يحتوي التقرير على مواصفات النظام وإصدارات المكتبات وحالة GPU، ولا يحتوي على Hugging Face Token.

In [ ]:
import json, platform, sys, os
from datetime import datetime, timezone
from pathlib import Path

gpu_info = []
cuda_available = False
cuda_version = None
torch_version = None
try:
    import torch
    torch_version = torch.__version__
    cuda_available = torch.cuda.is_available()
    cuda_version = torch.version.cuda
    if cuda_available:
        for i in range(torch.cuda.device_count()):
            p = torch.cuda.get_device_properties(i)
            free_b, total_b = torch.cuda.mem_get_info(i)
            gpu_info.append({
                "index": i,
                "name": p.name,
                "total_vram_gib": round(p.total_memory / (1024**3), 2),
                "free_vram_gib_at_report": round(free_b / (1024**3), 2),
                "compute_capability": f"{p.major}.{p.minor}"
            })
except Exception as exc:
    gpu_info = [{"error": repr(exc)}]

report = {
    "project": "Game Intelligence AI",
    "notebook": "01_environment_setup.ipynb",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "python": sys.version,
    "platform": platform.platform(),
    "colab": bool(os.environ.get("COLAB_RELEASE_TAG")),
    "torch_version": str(torch_version) if torch_version is not None else None,
    "cuda_available": cuda_available,
    "torch_cuda_build": str(cuda_version) if cuda_version is not None else None,
    "gpus": gpu_info,
    "packages": package_status,
    "imports": import_status,
    "model_repository_checked": MODEL_ID,
    "huggingface_token_configured": bool(HF_TOKEN),
    "drive_project_root": PROJECT_ROOT
}

docs_dir = Path(PROJECT_ROOT) / "documentation"
docs_dir.mkdir(parents=True, exist_ok=True)
json_path = docs_dir / "environment_report.json"
txt_path = docs_dir / "environment_report.txt"

json_path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")

summary_lines = [
    "GAME INTELLIGENCE AI — ENVIRONMENT REPORT",
    f"Created UTC: {report['created_utc']}",
    f"Python: {report['python']}",
    f"Platform: {report['platform']}",
    f"PyTorch: {report['torch_version']}",
    f"CUDA available: {report['cuda_available']}",
    f"PyTorch CUDA build: {report['torch_cuda_build']}",
    f"Model repository checked: {MODEL_ID}",
    "",
    "GPUs:"
]
summary_lines += [json.dumps(g, ensure_ascii=False) for g in gpu_info] or ["No GPU detected."]
summary_lines += ["", "Packages:"]
summary_lines += [f"{k}: {v}" for k, v in package_status.items()]
summary_lines += ["", "Imports:"]
summary_lines += [f"{k}: {v}" for k, v in import_status.items()]
txt_path.write_text("\n".join(summary_lines), encoding="utf-8")

print("Environment report saved:")
print(json_path)
print(txt_path)
print("\nGPU readiness:", "GPU DETECTED" if cuda_available else "NO GPU — setup can continue, but model training must wait.")

## الخلية 11 — قائمة تحقق نهائية

تُظهر ملخصًا واضحًا لما يجب أن يكون جاهزًا قبل الانتقال إلى Notebook المقارنة. لا تعتبر عدم توفر GPU فشلًا في ربط Drive أو إعداد الحساب؛ لكنه يمنعنا من بدء تدريب النموذج في هذه الجلسة.

In [ ]:
checks = {
    "Google Drive project folder exists": os.path.isdir(PROJECT_ROOT),
    "Environment report JSON exists": os.path.isfile(os.path.join(PROJECT_ROOT, "documentation", "environment_report.json")),
    "Environment report TXT exists": os.path.isfile(os.path.join(PROJECT_ROOT, "documentation", "environment_report.txt")),
    "PyTorch import": import_status.get("torch") == "OK",
    "Transformers import": import_status.get("transformers") == "OK",
    "PEFT import": import_status.get("peft") == "OK",
    "Hugging Face Hub import": import_status.get("huggingface_hub") == "OK",
}

for name, passed in checks.items():
    print(("PASS" if passed else "CHECK") + " — " + name)

print("\nGPU available:", cuda_available)
if all(checks.values()) and cuda_available:
    print("\nREADY: Environment setup passed. Next: 02_model_benchmark.ipynb")
elif all(checks.values()):
    print("\nPARTIAL READY: Drive and libraries are ready. Enable a GPU before model loading/training.")
else:
    print("\nNOT READY: Review the failed checks and the error messages above before continuing.")

## بعد الانتهاء — ماذا تفعل؟

1. تأكد أن الخلية الأخيرة أظهرت `READY` أو `PARTIAL READY`.
2. افتح Google Drive ثم مجلد `GameIntelligenceAI`.
3. تأكد من وجود `documentation/environment_report.json` و`environment_report.txt`.
4. لا تحذف التقرير؛ سنستخدمه لتحديد إعدادات Notebook التدريب وفق GPU الحقيقي.
5. احتفظ بهذا الـNotebook في مجلد `notebooks` داخل Drive أو ارفع نسخة منه إلى مستودع GitHub الخاص.

**الخطوة التالية:** `02_model_benchmark.ipynb` لاختبار النماذج المرشحة على مهام تطوير ألعاب موحدة قبل اتخاذ قرار التدريب.